# Bonus ablation: pretrain without positional embeddings

The Phase 2 pretraining run with one change: the positional-embedding table is
removed (`use_positional_embeddings=False`, 196,608 fewer parameters). Data,
the rest of the architecture, optimizer, learning-rate schedule, seed and step
budget all match `phase2_pretrain.ipynb`, so any difference from the standard
model is attributable to position information. Run it for **one** language;
Hindi is the suggested choice.

Checkpoints go to `<language>_nopos` and the resume dataset is named
`lma-bonus-<language>-nopos-checkpoint`, so this run can never resume from, or
be mistaken for, the standard model's `<language>_model` checkpoints.

### Before running

* Accelerator: **GPU** (P100 or T4). Settings -> Accelerator.
* Add the tokenized data: **+ Add Data** -> search your datasets ->
  `lma-phase2-tokenized`.
* On the second and later sessions, also add your own checkpoint dataset so the
  run continues instead of starting over. See the resume section at the bottom.

### Why this is split across sessions

Kaggle stops a session after 12 hours. A full run needs more than that, so the
trainer checkpoints every 1,000 steps and stops cleanly before the limit. A fresh
session resumes from the last checkpoint and continues on exactly the same
trajectory - this is verified bit-exact by `scripts/test_model.py`.

## 1. Choose the language

`hi` is Model H (Hindi, higher-resource); `ne` is Model L (Nepali).

In [ ]:
LANGUAGE = "hi"        # "hi" or "ne"
TIME_BUDGET_HOURS = 10.5   # stop and checkpoint before Kaggle's 12-hour cutoff

assert LANGUAGE in ("hi", "ne")
print(f"training Model {'H (Hindi)' if LANGUAGE == 'hi' else 'L (Nepali)'}")

## 2. The project code

The `lma` package travels inside this notebook rather than being cloned: the
GitHub Classroom repository is private, so an unauthenticated `git clone` from
Kaggle fails.

The cells below hold the model and training code exactly as committed - the
Transformer, the batching, and the training loop - written to `/kaggle/working/src` and
imported from there. They are plain readable Python rather than an encoded blob,
so everything this notebook runs can be read here.

In [ ]:
import sys
from pathlib import Path

SRC = Path("/kaggle/working/src")
for package in ("lma", "lma/model", "lma/train"):
    (SRC / package).mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"module directory ready: {SRC}")

In [ ]:
# Empty package markers. These cannot use %%writefile, which errors on
# an empty cell body, so they are created directly.
(SRC / "lma/__init__.py").touch()

print("created 1 empty package marker(s)")

In [ ]:
%%writefile /kaggle/working/src/lma/model/__init__.py
"""Decoder-only Transformer implemented from primitive PyTorch layers.

Phase 2 of the project forbids ``nn.Transformer*``, HuggingFace model classes,
and any pre-built attention block, so every tensor operation in the forward pass
is written out explicitly here.
"""

from .config import ModelConfig
from .transformer import (
    CausalSelfAttention,
    DecoderBlock,
    FeedForward,
    TransformerLM,
)

__all__ = [
    "ModelConfig",
    "CausalSelfAttention",
    "FeedForward",
    "DecoderBlock",
    "TransformerLM",
]

In [ ]:
%%writefile /kaggle/working/src/lma/model/config.py
"""Model configuration, with an exact parameter count derived from the shapes.

The assignment fixes a target of roughly 25M trainable parameters per model and
asks for the exact count to be reported, so the arithmetic lives here next to the
hyperparameters rather than being measured only after a model is built. That also
makes it possible to search for a configuration that hits the budget without
allocating anything.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field


@dataclass
class ModelConfig:
    """Hyperparameters for one decoder-only language model.

    The two models in this project share an architecture but not a vocabulary:
    each uses the tokenizer trained on its own corpus in Phase 1, so
    `vocab_size` differs and the embedding and output layers differ in size.

    Attributes:
        vocab_size: Number of tokenizer pieces. From the Phase 1 tokenizer.
        n_layer: Number of stacked decoder blocks.
        n_head: Number of attention heads. Must divide `n_embd`.
        n_embd: Model dimension `d`.
        n_inner: Inner width of the feed-forward network. Defaults to 4x `n_embd`,
            the usual ratio, and the assignment requires it to exceed `n_embd`.
        block_size: Maximum context length in tokens. Learned positional
            embeddings are allocated for exactly this many positions, which is
            the hard limit on sequence length at inference (see `TransformerLM`).
        dropout: Applied to embeddings, attention probabilities, and sublayer
            outputs.
        tie_weights: Share the input embedding matrix with the output projection.
            Saves `vocab_size * n_embd` parameters, which at these sizes is a
            fifth of the whole budget.
        bias: Whether linear layers carry bias terms.
        use_positional_embeddings: Whether position is injected at all. Setting
            this to ``False`` is the assignment's optional ablation. Without it, a
            single attention layer sees the tokens before each position as an
            unordered set and cannot tell their orderings apart. The causal mask
            still leaks order into deeper layers - each position's state
            summarizes a prefix of a different length - so how much of the lost
            signal a deep stack recovers is what the ablation measures. Kept as a
            flag so the ablated model is the same code path, and so a checkpoint
            records which variant produced it.
    """

    vocab_size: int
    n_layer: int = 12
    n_head: int = 6
    n_embd: int = 384
    n_inner: int | None = None
    block_size: int = 512
    dropout: float = 0.1
    tie_weights: bool = True
    bias: bool = True
    use_positional_embeddings: bool = True

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError(
                f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})"
            )
        if self.n_inner is None:
            self.n_inner = 4 * self.n_embd
        if self.n_inner <= self.n_embd:
            raise ValueError(
                f"n_inner ({self.n_inner}) must exceed n_embd ({self.n_embd}); the "
                "feed-forward network is required to project up before projecting back"
            )

    @property
    def head_dim(self) -> int:
        """Per-head subspace width `d_k = d / h`."""
        return self.n_embd // self.n_head

    def parameter_count(self) -> dict[str, int]:
        """Exact trainable-parameter count, broken down by component.

        Computed from shapes rather than by instantiating the model, so a
        configuration can be checked against the 25M budget cheaply.

        Returns:
            Counts per component plus a ``total`` key.
        """
        d, v, ff = self.n_embd, self.vocab_size, self.n_inner
        b = 1 if self.bias else 0

        token_embedding = v * d
        # The ablation removes the positional table entirely rather than zeroing
        # it, so the parameter count reflects what is actually trained.
        position_embedding = self.block_size * d if self.use_positional_embeddings else 0

        # Per block: four d x d projections (Q, K, V, O) and two feed-forward
        # matrices, plus two LayerNorms (weight and bias each).
        attn = 4 * (d * d + b * d)
        mlp = (d * ff + b * ff) + (ff * d + b * d)
        norms = 2 * (2 * d)
        per_block = attn + mlp + norms
        blocks = self.n_layer * per_block

        final_norm = 2 * d
        # The output projection is free when tied to the input embedding.
        head = 0 if self.tie_weights else v * d

        total = token_embedding + position_embedding + blocks + final_norm + head
        return {
            "token_embedding": token_embedding,
            "position_embedding": position_embedding,
            "blocks": blocks,
            "per_block": per_block,
            "final_norm": final_norm,
            "lm_head": head,
            "total": total,
        }

    @property
    def n_params(self) -> int:
        """Total trainable parameters."""
        return self.parameter_count()["total"]

    def embedding_share(self) -> float:
        """Fraction of the budget spent on embeddings rather than on layers.

        The quantity the Phase 1 vocabulary sweep was chosen against: every piece
        added to the vocabulary is a parameter taken away from the blocks that do
        the modelling.
        """
        counts = self.parameter_count()
        embeddings = counts["token_embedding"] + counts["position_embedding"]
        return embeddings / counts["total"]

    def to_dict(self) -> dict:
        """Serialize for checkpointing and for the run's config record."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "ModelConfig":
        """Rebuild from a checkpoint payload, ignoring unknown keys."""
        fields = {f for f in cls.__dataclass_fields__}
        return cls(**{k: v for k, v in data.items() if k in fields})

In [ ]:
%%writefile /kaggle/working/src/lma/model/transformer.py
"""A decoder-only (GPT-style) Transformer written from primitive layers.

Only ``nn.Linear``, ``nn.Embedding``, ``nn.LayerNorm``, and ``nn.Dropout`` are
used. Attention is computed explicitly - the projection, the head reshape, the
scaled dot product, the causal mask, the softmax, and the output projection are
all written out - because the point of this phase is to be able to explain every
tensor operation in the forward pass.

Shape convention throughout: ``B`` batch, ``T`` time (sequence position), ``d``
model dimension, ``h`` heads, ``dk = d / h`` per-head width.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import ModelConfig


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention.

    Each position attends to itself and to earlier positions only. The mask is
    additive and applied to the scores *before* the softmax, so masked positions
    receive zero probability rather than being renormalized afterwards.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.head_dim

        # One matrix producing Q, K and V together. Mathematically identical to
        # three separate d x d projections, but a single matmul is faster.
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Upper-triangular mask, held as a buffer so it moves with .to(device)
        # and is saved in no gradient. Bool: True where attention is forbidden.
        mask = torch.triu(
            torch.ones(config.block_size, config.block_size, dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Apply causal self-attention.

        Args:
            x: Input activations, shape ``(B, T, d)``.
            return_attention: Also return the post-softmax attention
                probabilities, shape ``(B, h, T, T)``. Used by the Phase 2
                attention analysis (heatmaps, entropy, mean distance); left off
                during training because materializing them costs memory.

        Returns:
            ``(output, attention)`` where output has shape ``(B, T, d)`` and
            attention is ``None`` unless requested.
        """
        B, T, d = x.shape

        # Project to queries, keys and values: (B, T, d) -> (B, T, 3d) -> 3 x (B, T, d)
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)

        # Split each into heads and move the head axis in front of time, so that
        # every head is an independent (T, dk) sequence for the matmul below:
        #   (B, T, d) -> (B, T, h, dk) -> (B, h, T, dk)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product scores: (B, h, T, dk) @ (B, h, dk, T) -> (B, h, T, T).
        #
        # The 1/sqrt(dk) scaling matters. q and k have unit-ish variance per
        # component, so their dot product over dk dimensions has variance
        # proportional to dk. Without the scaling, scores grow with model width,
        # the softmax saturates, and its gradient vanishes - training stalls for
        # exactly the wider models the scaling was meant to enable.
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask: -inf on future positions, so softmax assigns them exactly
        # zero. Sliced to T because the final batch of a sequence may be shorter.
        scores = scores.masked_fill(self.causal_mask[:T, :T], float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        # Weighted sum of values: (B, h, T, T) @ (B, h, T, dk) -> (B, h, T, dk)
        y = attn @ v

        # Concatenate heads back together: (B, h, T, dk) -> (B, T, h, dk) -> (B, T, d).
        # contiguous() because transpose leaves a non-contiguous view that view()
        # cannot reinterpret.
        y = y.transpose(1, 2).contiguous().view(B, T, d)

        y = self.resid_dropout(self.proj(y))
        return y, (attn if return_attention else None)


class FeedForward(nn.Module):
    """Position-wise feed-forward network.

    Two linear layers with a GELU between them, applied identically at every
    position. The inner dimension is wider than the model dimension (4x by
    default): this is where most of the block's capacity lives, and where the
    model does per-token computation that attention - which only mixes
    information between positions - cannot do.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.fc = nn.Linear(config.n_embd, config.n_inner, bias=config.bias)
        self.proj = nn.Linear(config.n_inner, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map ``(B, T, d)`` to ``(B, T, d)`` through the wider inner layer."""
        return self.dropout(self.proj(F.gelu(self.fc(x))))


class DecoderBlock(nn.Module):
    """One pre-norm decoder block: attention, then feed-forward, both residual.

    Pre-norm (normalize *entering* each sublayer, add the raw residual) rather
    than post-norm. With pre-norm the residual path from input to loss carries no
    normalization, so gradients reach early layers undiminished and deep stacks
    train without a warmup-dependent divergence. Post-norm places a LayerNorm on
    that path and needs careful warmup to remain stable at this depth.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Run both sublayers, each wrapped in a residual connection."""
        attn_out, attn_weights = self.attn(self.ln_1(x), return_attention)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, attn_weights


class TransformerLM(nn.Module):
    """Decoder-only Transformer language model.

    Learned absolute positional embeddings are used rather than sinusoidal or
    rotary. The reason is that this model is trained and evaluated at a single
    fixed context length, where learned positions are simplest and train fastest;
    the cost is that the model cannot be run on sequences longer than
    `block_size`, because no embedding exists for those positions. Sinusoidal or
    RoPE encodings would extrapolate, at the price of extra machinery this
    project does not need.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        # The ablation omits the table entirely rather than zeroing it, so the
        # parameters are genuinely absent from the model and from its count.
        self.position_embedding = (
            nn.Embedding(config.block_size, config.n_embd)
            if config.use_positional_embeddings
            else None
        )
        self.embed_dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(DecoderBlock(config) for _ in range(config.n_layer))
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        if config.tie_weights:
            # One matrix serving as both the input embedding and the output
            # projection. Saves vocab_size x n_embd parameters - about a fifth of
            # this budget - and ties the representation of a token to the vector
            # that predicts it.
            self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

        # Scaled init for the residual projections. Each block adds into the
        # residual stream, so with n_layer blocks the variance accumulated at the
        # top grows with depth; scaling these by 1/sqrt(2 * n_layer) keeps
        # activations stable at initialization regardless of depth.
        for name, param in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(
                    param, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer)
                )

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        """Normal(0, 0.02) for weights, zeros for biases - the GPT-2 scheme."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_parameters(self, trainable_only: bool = True) -> int:
        """Count parameters, counting a tied embedding matrix only once."""
        seen: set[int] = set()
        total = 0
        for param in self.parameters():
            if trainable_only and not param.requires_grad:
                continue
            if id(param) in seen:
                continue
            seen.add(id(param))
            total += param.numel()
        return total

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor | None = None,
        return_attention: bool = False,
    ) -> dict:
        """Predict next-token logits, and optionally the language-modeling loss.

        Args:
            idx: Token ids, shape ``(B, T)``.
            targets: Next-token ids, shape ``(B, T)``. When given, the causal
                language-modeling loss is returned alongside the logits.
            return_attention: Collect per-layer attention probabilities.

        Returns:
            Dict with ``logits`` ``(B, T, vocab_size)``, ``loss`` (or None), and
            ``attentions`` (list of ``(B, h, T, T)`` tensors, or None).
        """
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError(
                f"sequence length {T} exceeds block_size {self.config.block_size}; "
                "learned positional embeddings exist only for block_size positions"
            )

        # Token identity plus position. Addition, not concatenation: the two
        # signals share the same d dimensions, and the model learns to separate
        # them. Without the positional term, one attention layer sees the tokens
        # before each position as an unordered set, so on its own it cannot tell
        # "dog bites man" from "man bites dog". The causal mask leaks some order
        # into deeper layers, and the positional-embedding ablation measures how
        # much of that lost signal a deep stack recovers.
        x = self.token_embedding(idx)
        if self.position_embedding is not None:
            pos = torch.arange(T, device=idx.device)
            x = x + self.position_embedding(pos)
        x = self.embed_dropout(x)

        attentions = [] if return_attention else None
        for block in self.blocks:
            x, attn = block(x, return_attention)
            if return_attention:
                attentions.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Cross-entropy between the logits at position t and the token at
            # t+1, averaged over every position in the batch. The caller supplies
            # targets already shifted by one.
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=-100,
            )

        return {"logits": logits, "loss": loss, "attentions": attentions}

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        greedy: bool = False,
        eos_id: int | None = None,
    ) -> torch.Tensor:
        """Autoregressively extend a prompt.

        Args:
            idx: Prompt token ids, shape ``(B, T)``.
            max_new_tokens: How many tokens to append.
            temperature: Divides the logits before softmax. Below 1 sharpens the
                distribution, above 1 flattens it. Ignored when `greedy`.
            top_k: Restrict sampling to the k most likely tokens.
            greedy: Take the argmax instead of sampling.
            eos_id: Stop early once every sequence has produced this token.

        Returns:
            The prompt with the generated continuation appended.
        """
        self.eval()
        finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)

        for _ in range(max_new_tokens):
            # Crop to the context window: positions beyond block_size have no
            # learned embedding, so the model can only condition on the most
            # recent block_size tokens.
            idx_cond = idx[:, -self.config.block_size :]
            logits = self(idx_cond)["logits"][:, -1, :]

            if greedy:
                next_token = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-8)
                if top_k is not None:
                    kth = logits.topk(min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                next_token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)

            if eos_id is not None:
                # Once a sequence has emitted EOS, keep emitting it so the tensor
                # stays rectangular; the caller truncates at the first EOS.
                next_token = torch.where(
                    finished.unsqueeze(1),
                    torch.full_like(next_token, eos_id),
                    next_token,
                )
                finished |= next_token.squeeze(1) == eos_id

            idx = torch.cat([idx, next_token], dim=1)
            if eos_id is not None and bool(finished.all()):
                break

        return idx

In [ ]:
%%writefile /kaggle/working/src/lma/train/__init__.py
"""Pretraining: batching, optimization, and resumable checkpoints."""

from .data import TokenBinaryDataset
from .trainer import TrainConfig, Trainer, TrainState, build_optimizer, cosine_lr

__all__ = [
    "TokenBinaryDataset",
    "TrainConfig",
    "Trainer",
    "TrainState",
    "build_optimizer",
    "cosine_lr",
]

In [ ]:
%%writefile /kaggle/working/src/lma/train/data.py
"""Batching for causal language-model pretraining.

The corpus is one flat ``uint16`` array of token ids on disk, produced by the
tokenization step. Batches are random windows into that array, memory-mapped so
the file is never loaded whole - a 1.2 GB corpus streams from page cache while
resident memory stays near zero.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import torch


class TokenBinaryDataset:
    """Random fixed-length windows over a flat array of token ids.

    Two details that matter for correctness:

    * **Targets are inputs shifted by one.** A window of ``block_size + 1``
      tokens yields ``x = w[:-1]`` and ``y = w[1:]``, so position ``t`` predicts
      the token at ``t + 1``. Sampling ``block_size + 1`` rather than
      ``block_size`` avoids the off-by-one that silently trains the model to
      predict its own input.
    * **Windows may cross document boundaries.** Documents were concatenated with
      an EOS token between them, so the model sees EOS as an ordinary token and
      learns that text ends there. This is standard for pretraining and means no
      document structure has to be tracked at batch time.
    """

    def __init__(self, path: str | Path, block_size: int, seed: int = 0) -> None:
        """Open a token binary for sampling.

        Args:
            path: ``.bin`` file of ``uint16`` token ids.
            block_size: Context length. Windows are ``block_size + 1`` long.
            seed: Seeds this dataset's own generator, so batch order is
                reproducible and independent of global torch/numpy state.
        """
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"token binary not found: {self.path}")

        self.block_size = block_size
        # mode="r" keeps the file read-only and lets several workers - or both
        # languages' loaders - share the same page cache.
        self.tokens = np.memmap(self.path, dtype=np.uint16, mode="r")
        self.rng = np.random.default_rng(seed)

        if len(self.tokens) <= block_size + 1:
            raise ValueError(
                f"{self.path.name} holds {len(self.tokens):,} tokens, too few for "
                f"block_size {block_size}"
            )

    def __len__(self) -> int:
        """Number of distinct starting positions."""
        return len(self.tokens) - self.block_size - 1

    def batch(self, batch_size: int, device: str = "cpu") -> tuple[torch.Tensor, torch.Tensor]:
        """Draw one batch of random windows.

        Returns:
            ``(x, y)``, both ``(batch_size, block_size)`` int64 tensors, with
            ``y`` shifted one position ahead of ``x``.
        """
        starts = self.rng.integers(0, len(self), size=batch_size)

        # Build in numpy, convert once: per-sample torch indexing is markedly
        # slower than one contiguous copy.
        window = np.stack(
            [self.tokens[s : s + self.block_size + 1].astype(np.int64) for s in starts]
        )
        batch = torch.from_numpy(window)

        x = batch[:, :-1].contiguous()
        y = batch[:, 1:].contiguous()

        if device != "cpu":
            # non_blocking with pinned memory overlaps the host-to-device copy
            # with compute on the previous batch.
            x = x.pin_memory().to(device, non_blocking=True)
            y = y.pin_memory().to(device, non_blocking=True)
        return x, y

    def sequential_batches(
        self, batch_size: int, limit: int | None = None, device: str = "cpu"
    ):
        """Iterate the corpus in order, for deterministic evaluation.

        Validation loss should not depend on a random draw, so held-out
        evaluation walks contiguous non-overlapping windows from the start of the
        file instead of sampling.

        Args:
            batch_size: Windows per batch.
            limit: Stop after this many batches. ``None`` walks the whole split.
            device: Where to place the tensors.

        Yields:
            ``(x, y)`` pairs shaped as in :meth:`batch`.
        """
        stride = self.block_size
        position = 0
        produced = 0

        while position + (batch_size * stride) + 1 <= len(self.tokens):
            if limit is not None and produced >= limit:
                return

            starts = [position + i * stride for i in range(batch_size)]
            window = np.stack(
                [self.tokens[s : s + self.block_size + 1].astype(np.int64) for s in starts]
            )
            batch = torch.from_numpy(window)

            x = batch[:, :-1].contiguous()
            y = batch[:, 1:].contiguous()
            if device != "cpu":
                x = x.pin_memory().to(device, non_blocking=True)
                y = y.pin_memory().to(device, non_blocking=True)

            yield x, y
            position += batch_size * stride
            produced += 1

    def get_rng_state(self) -> dict:
        """Sampler RNG state, for checkpointing.

        The dataset draws window offsets from its own generator rather than the
        global one, so resuming a run has to restore this too - otherwise a
        resumed run sees a different batch sequence than an uninterrupted one and
        the two diverge.
        """
        return self.rng.bit_generator.state

    def set_rng_state(self, state: dict) -> None:
        """Restore a sampler RNG state saved by :meth:`get_rng_state`."""
        self.rng.bit_generator.state = state

    def stats(self) -> dict:
        """Summary for the run log."""
        return {
            "path": str(self.path),
            "tokens": int(len(self.tokens)),
            "bytes": self.path.stat().st_size,
            "windows": len(self),
            "block_size": self.block_size,
        }

In [ ]:
%%writefile /kaggle/working/src/lma/train/trainer.py
"""Pretraining loop for the Phase 2 language models.

Built around one constraint that shapes every design choice here: Kaggle kills a
session after 12 hours, and a full run needs more than that. So training is
resumable at any step - checkpoints carry model weights, optimizer state,
scheduler position, step counter, RNG state, and the config - and a fresh session
picks up exactly where the last one stopped rather than restarting.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import torch
import torch.nn as nn

from ..model import ModelConfig, TransformerLM
from .data import TokenBinaryDataset


@dataclass
class TrainConfig:
    """Optimization hyperparameters for one pretraining run."""

    batch_size: int = 24
    gradient_accumulation: int = 4
    learning_rate: float = 6e-4
    min_learning_rate: float = 6e-5
    warmup_steps: int = 500
    max_steps: int = 12_000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    eval_interval: int = 500
    eval_batches: int = 100
    checkpoint_interval: int = 1_000
    log_interval: int = 20
    seed: int = 20260812
    amp: bool = True
    compile: bool = False

    @property
    def tokens_per_step(self) -> int:
        """Tokens consumed per optimizer step, ignoring block_size."""
        return self.batch_size * self.gradient_accumulation

    def to_dict(self) -> dict:
        """Serialize for checkpointing and for the run's config record."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "TrainConfig":
        """Rebuild from a checkpoint payload, ignoring unknown keys.

        Unknown keys are dropped rather than raising, so a checkpoint written by
        an earlier version - before a field existed, or after one was removed -
        still loads instead of stranding a trained model.
        """
        fields = set(cls.__dataclass_fields__)
        return cls(**{k: v for k, v in data.items() if k in fields})


def cosine_lr(step: int, cfg: TrainConfig) -> float:
    """Learning rate at `step`: linear warmup, then cosine decay.

    Warmup exists because Adam's second-moment estimate is unreliable in the
    first few hundred steps; starting at full learning rate there produces the
    large early updates that destabilize deep pre-norm stacks. Cosine decay
    afterwards spends most of the budget at a high rate and anneals smoothly,
    which reliably beats a constant rate at a fixed step count.
    """
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / max(cfg.warmup_steps, 1)

    progress = (step - cfg.warmup_steps) / max(cfg.max_steps - cfg.warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.min_learning_rate + coeff * (cfg.learning_rate - cfg.min_learning_rate)


def build_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.AdamW:
    """AdamW with weight decay applied only where it belongs.

    Decaying biases, LayerNorm gains, and embeddings hurts: those parameters set
    scale and offset rather than mixing features, and shrinking them toward zero
    is a bias with no regularizing benefit. Only matrices that participate in
    matrix multiplications - the 2D weights - are decayed.
    """
    decay, no_decay = [], []
    seen: set[int] = set()

    for module in model.modules():
        for name, param in module.named_parameters(recurse=False):
            if not param.requires_grad or id(param) in seen:
                continue
            seen.add(id(param))
            if param.dim() >= 2 and not isinstance(module, nn.Embedding):
                decay.append(param)
            else:
                no_decay.append(param)

    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    # fused AdamW is markedly faster on CUDA and a no-op elsewhere.
    fused = torch.cuda.is_available()
    return torch.optim.AdamW(
        groups, lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), fused=fused
    )


def _as_byte_tensor(state) -> torch.Tensor:
    """Coerce a saved RNG state into the CPU uint8 tensor torch requires.

    ``torch.set_rng_state`` and ``torch.cuda.set_rng_state_all`` accept only a
    ByteTensor on CPU, and raise ``TypeError: RNG state must be a
    torch.ByteTensor`` otherwise. A state that has been through
    ``torch.save``/``torch.load`` - especially across torch versions or between
    hosts - does not always come back in that exact form.
    """
    if not isinstance(state, torch.Tensor):
        state = torch.tensor(state, dtype=torch.uint8)
    return state.cpu().to(torch.uint8)


@dataclass
class TrainState:
    """Everything needed to resume a run mid-flight."""

    step: int = 0
    tokens_seen: int = 0
    best_val_loss: float = float("inf")
    history: list[dict] = field(default_factory=list)
    wall_seconds: float = 0.0


class Trainer:
    """Pretrain one language model, resumably.

    Args:
        model: The model to train.
        train_data: Sampler over the training binary.
        val_data: Sampler over the validation binary.
        cfg: Optimization hyperparameters.
        out_dir: Where checkpoints and the run log are written.
        device: ``"cuda"`` or ``"cpu"``.
    """

    def __init__(
        self,
        model: TransformerLM,
        train_data: TokenBinaryDataset,
        val_data: TokenBinaryDataset,
        cfg: TrainConfig,
        out_dir: str | Path,
        device: str | None = None,
    ) -> None:
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_data = train_data
        self.val_data = val_data
        self.cfg = cfg
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

        self.optimizer = build_optimizer(self.model, cfg)
        self.state = TrainState()

        # Mixed precision: bfloat16 where supported (no loss scaling needed,
        # same exponent range as fp32), otherwise fp16 with a gradient scaler to
        # keep small gradients from flushing to zero.
        self.amp_enabled = cfg.amp and self.device == "cuda"
        self.amp_dtype = torch.float32
        if self.amp_enabled:
            supported = torch.cuda.is_bf16_supported()
            self.amp_dtype = torch.bfloat16 if supported else torch.float16
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=self.amp_enabled and self.amp_dtype is torch.float16
        )

        if cfg.compile and hasattr(torch, "compile"):
            self.model = torch.compile(self.model)

        torch.manual_seed(cfg.seed)

    # -- evaluation ---------------------------------------------------------

    @torch.no_grad()
    def evaluate(self, batches: int | None = None) -> dict:
        """Mean cross-entropy over held-out data, and the perplexity implied.

        Walks the validation split in order rather than sampling, so the number
        is comparable across steps and across runs.
        """
        self.model.eval()
        n_batches = batches if batches is not None else self.cfg.eval_batches

        losses = []
        for x, y in self.val_data.sequential_batches(
            self.cfg.batch_size, limit=n_batches, device=self.device
        ):
            with torch.autocast(
                device_type=self.device, dtype=self.amp_dtype, enabled=self.amp_enabled
            ):
                loss = self.model(x, targets=y)["loss"]
            losses.append(loss.float().item())

        self.model.train()
        mean_loss = sum(losses) / max(len(losses), 1)
        return {
            "val_loss": mean_loss,
            # Perplexity is exp of the mean cross-entropy, per the assignment.
            "val_perplexity": math.exp(min(mean_loss, 20)),
            "batches": len(losses),
        }

    # -- checkpointing ------------------------------------------------------

    def save_checkpoint(self, name: str = "latest") -> Path:
        """Write a checkpoint that a fresh session can resume from.

        Carries model weights, optimizer state, the step counter, and the config,
        as the assignment requires - plus RNG state, so a resumed run draws the
        same batch sequence it would have drawn had it never stopped. The
        learning-rate schedule is a pure function of the step, so storing the step
        stores the scheduler position.
        """
        model = getattr(self.model, "_orig_mod", self.model)  # unwrap torch.compile
        payload = {
            "model": model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
            "step": self.state.step,
            "tokens_seen": self.state.tokens_seen,
            "best_val_loss": self.state.best_val_loss,
            "history": self.state.history,
            "wall_seconds": self.state.wall_seconds,
            "model_config": model.config.to_dict(),
            "train_config": self.cfg.to_dict(),
            # The sampler's own generator, without which a resumed run draws a
            # different batch sequence and diverges from an uninterrupted one.
            "train_sampler_rng": self.train_data.get_rng_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        path = self.out_dir / f"checkpoint_{name}.pt"
        # Write to a temporary file first: a session killed mid-write would
        # otherwise leave a truncated checkpoint and lose the whole run.
        tmp = path.with_suffix(".pt.tmp")
        torch.save(payload, tmp)
        tmp.replace(path)
        return path

    def load_checkpoint(self, path: str | Path) -> None:
        """Restore a run from a checkpoint written by :meth:`save_checkpoint`."""
        payload = torch.load(path, map_location=self.device, weights_only=False)
        model = getattr(self.model, "_orig_mod", self.model)
        model.load_state_dict(payload["model"])
        self.optimizer.load_state_dict(payload["optimizer"])
        if payload.get("scaler"):
            self.scaler.load_state_dict(payload["scaler"])

        self.state = TrainState(
            step=payload["step"],
            tokens_seen=payload["tokens_seen"],
            best_val_loss=payload["best_val_loss"],
            history=payload.get("history", []),
            wall_seconds=payload.get("wall_seconds", 0.0),
        )

        # Restoring RNG state makes a resumed run follow the same trajectory as an
        # uninterrupted one. It is a fidelity improvement, not a correctness
        # requirement: the model, optimizer, and step above are what actually
        # continue the run. So every restore below is best-effort - a checkpoint
        # written under a different torch version or GPU count can carry states
        # this build will not accept, and refusing to resume over that would
        # throw away a valid checkpoint for no benefit.
        if payload.get("train_sampler_rng") is not None:
            try:
                self.train_data.set_rng_state(payload["train_sampler_rng"])
            except (ValueError, KeyError, TypeError) as error:
                print(f"  note: sampler RNG state not restored ({error})")

        if payload.get("torch_rng") is not None:
            try:
                torch.set_rng_state(_as_byte_tensor(payload["torch_rng"]))
            except (RuntimeError, TypeError) as error:
                print(f"  note: CPU RNG state not restored ({error})")

        if payload.get("cuda_rng") and torch.cuda.is_available():
            try:
                states = [_as_byte_tensor(state) for state in payload["cuda_rng"]]
                # set_rng_state_all expects exactly one state per visible device;
                # a checkpoint from a host with a different GPU count would
                # otherwise raise deep inside the CUDA lazy-init callback.
                if len(states) == torch.cuda.device_count():
                    torch.cuda.set_rng_state_all(states)
                else:
                    print(
                        f"  note: CUDA RNG state not restored "
                        f"({len(states)} saved, {torch.cuda.device_count()} device(s) here)"
                    )
            except (RuntimeError, TypeError) as error:
                print(f"  note: CUDA RNG state not restored ({error})")

    def find_resume_point(self) -> Path | None:
        """Latest checkpoint in the output directory, if any."""
        latest = self.out_dir / "checkpoint_latest.pt"
        return latest if latest.exists() else None

    # -- training -----------------------------------------------------------

    def train(self, max_seconds: float | None = None, resume: bool = True) -> TrainState:
        """Run the training loop.

        Args:
            max_seconds: Stop cleanly after this much wall time, saving a
                checkpoint first. Set it below the platform's session limit so
                the run ends on its own terms rather than being killed mid-step.
            resume: Continue from ``checkpoint_latest.pt`` when one exists.

        Returns:
            The final training state.
        """
        if resume:
            found = self.find_resume_point()
            if found:
                self.load_checkpoint(found)
                print(f"resumed from {found.name} at step {self.state.step:,}")

        cfg = self.cfg
        block = self.train_data.block_size
        self.model.train()
        started = time.time()
        window_start = time.time()
        window_loss = 0.0
        window_steps = 0

        while self.state.step < cfg.max_steps:
            lr = cosine_lr(self.state.step, cfg)
            for group in self.optimizer.param_groups:
                group["lr"] = lr

            # Gradient accumulation: several micro-batches summed into one
            # update, so the effective batch can exceed what fits in memory.
            self.optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0

            for _ in range(cfg.gradient_accumulation):
                x, y = self.train_data.batch(cfg.batch_size, device=self.device)
                with torch.autocast(
                    device_type=self.device, dtype=self.amp_dtype, enabled=self.amp_enabled
                ):
                    loss = self.model(x, targets=y)["loss"]
                    # Divide so the accumulated gradient is the mean over
                    # micro-batches, not their sum.
                    loss = loss / cfg.gradient_accumulation

                self.scaler.scale(loss).backward()
                step_loss += loss.float().item()

            if cfg.grad_clip > 0:
                # Unscale before clipping, or the threshold would apply to
                # scaled gradients and mean nothing.
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.grad_clip)

            self.scaler.step(self.optimizer)
            self.scaler.update()

            self.state.step += 1
            self.state.tokens_seen += cfg.tokens_per_step * block
            window_loss += step_loss
            window_steps += 1

            if self.state.step % cfg.log_interval == 0:
                elapsed = time.time() - window_start
                tps = (cfg.tokens_per_step * block * window_steps) / max(elapsed, 1e-9)
                print(
                    f"step {self.state.step:>6,}/{cfg.max_steps:,}  "
                    f"loss {window_loss / window_steps:.4f}  "
                    f"lr {lr:.2e}  {tps / 1e3:.1f}k tok/s  "
                    f"seen {self.state.tokens_seen / 1e6:.0f}M",
                    flush=True,
                )
                window_loss = 0.0
                window_steps = 0
                window_start = time.time()

            if self.state.step % cfg.eval_interval == 0:
                metrics = self.evaluate()
                metrics.update(step=self.state.step, lr=lr, tokens_seen=self.state.tokens_seen)
                self.state.history.append(metrics)
                print(
                    f"  eval step {self.state.step:,}: "
                    f"val_loss {metrics['val_loss']:.4f}  "
                    f"ppl {metrics['val_perplexity']:.2f}",
                    flush=True,
                )
                if metrics["val_loss"] < self.state.best_val_loss:
                    self.state.best_val_loss = metrics["val_loss"]
                    self.save_checkpoint("best")
                self._write_history()

            if self.state.step % cfg.checkpoint_interval == 0:
                self.save_checkpoint("latest")

            if max_seconds is not None and (time.time() - started) > max_seconds:
                self.state.wall_seconds += time.time() - started
                self.save_checkpoint("latest")
                self._write_history()
                print(
                    f"\nstopped at step {self.state.step:,} after "
                    f"{(time.time() - started) / 60:.0f} min (time budget reached); "
                    "rerun to continue from this checkpoint"
                )
                return self.state

        self.state.wall_seconds += time.time() - started
        self.save_checkpoint("latest")
        final = self.evaluate()
        final.update(step=self.state.step, tokens_seen=self.state.tokens_seen, final=True)
        self.state.history.append(final)
        self._write_history()
        print(
            f"\ntraining complete at step {self.state.step:,}: "
            f"val_loss {final['val_loss']:.4f}  ppl {final['val_perplexity']:.2f}"
        )
        return self.state

    def _write_history(self) -> None:
        """Persist the metric history for the report's loss curves."""
        path = self.out_dir / "training_history.json"
        path.write_text(
            json.dumps(
                {
                    "step": self.state.step,
                    "tokens_seen": self.state.tokens_seen,
                    "best_val_loss": self.state.best_val_loss,
                    "wall_seconds": round(self.state.wall_seconds, 1),
                    "train_config": self.cfg.to_dict(),
                    "history": self.state.history,
                },
                indent=1,
            )
        )

In [ ]:
# Confirm every module landed, and that the package imports.
import hashlib

for rel in ['lma/__init__.py', 'lma/model/__init__.py', 'lma/model/config.py', 'lma/model/transformer.py', 'lma/train/__init__.py', 'lma/train/data.py', 'lma/train/trainer.py']:
    text = (SRC / rel).read_text(encoding="utf-8")
    digest = hashlib.sha256(text.encode()).hexdigest()[:12]
    print(f"  {rel:32s} {len(text):>6,} bytes  sha256:{digest}")

from lma.model import ModelConfig, TransformerLM
from lma.train import TokenBinaryDataset, TrainConfig, Trainer

print("imports OK")

## 3. Locate the data and confirm the GPU

In [ ]:
import torch

INPUT = Path("/kaggle/input")

# rglob rather than a fixed depth: Kaggle nests a dataset under its own name, and
# the tokenization notebook wrote its output inside a "tokenized/" folder, so the
# binaries can sit two or three levels down depending on how the dataset was
# created. Searching recursively covers every arrangement.
candidates = sorted(INPUT.rglob("*_train.bin"))
if not candidates:
    available = sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else []
    raise FileNotFoundError(
        "No *_train.bin found under /kaggle/input. "
        f"Attached datasets: {available or 'none'}. "
        "Use '+ Add Data' to attach lma-phase2-tokenized."
    )

DATA_DIR = candidates[0].parent
print("data:", DATA_DIR)
for path in sorted(DATA_DIR.glob("*.bin")):
    print(f"   {path.stat().st_size / 1e9:6.2f} GB  {path.name}")

# Fail early and clearly if the split this run needs is missing, rather than
# deep inside the training loop.
for split in ("train", "validation"):
    expected = DATA_DIR / f"{LANGUAGE}_{split}.bin"
    if not expected.exists():
        raise FileNotFoundError(f"{expected} is missing from the attached dataset")

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"\nGPU: {properties.name}  {properties.total_memory / 1e9:.1f} GB")
    print(f"   compute capability: sm_{capability[0]}{capability[1]}")
    print(f"   torch {torch.__version__}, built for CUDA {torch.version.cuda}")
    print(f"   kernels compiled for: {torch.cuda.get_arch_list()}")
    print(f"   bf16 supported: {torch.cuda.is_bf16_supported()}")

    # A torch build carrying no kernels for this card fails with "no kernel image
    # is available for execution on the device" on the first real operation -
    # which happens inside the training loop, long after the setup cells have
    # reported success. Provoke it here instead, where the diagnosis is cheap.
    try:
        probe = torch.zeros(8, device="cuda") + 1
        _ = (probe @ probe).item()
        print("\n   GPU compute check: OK")
    except RuntimeError as error:
        raise RuntimeError(
            f"This PyTorch build has no kernels for sm_{capability[0]}{capability[1]} "
            f"(it carries {torch.cuda.get_arch_list()}). Switch the Accelerator to a "
            "different GPU type, or reinstall a matching torch build. "
            f"Original error: {error}"
        ) from error
else:
    print("\nWARNING: no GPU. Set Accelerator to GPU in Settings, or this will take days.")

## 4. Build the ablated model

The standard Phase 2 architecture with the positional-embedding table removed.
The assertion stops the run if the flag did not take effect, because training
the standard model by mistake would cost a whole session.

In [ ]:
from lma.model import ModelConfig, TransformerLM
from lma.train import TokenBinaryDataset, TrainConfig, Trainer

model_config = ModelConfig(
    vocab_size=8000,
    n_layer=12,
    n_head=6,
    n_embd=384,
    n_inner=1536,
    block_size=512,
    dropout=0.1,
    tie_weights=True,
    use_positional_embeddings=False,   # the ablation
)

model = TransformerLM(model_config)
assert model.position_embedding is None, "ablation flag did not take effect"
counts = model_config.parameter_count()

print(f"parameters: {model.num_parameters():,}")
for key in ("token_embedding", "position_embedding", "blocks", "final_norm"):
    print(f"   {key:20s} {counts[key]:>12,}")
print(f"   embedding share:     {100 * model_config.embedding_share():.1f}%")

## 5. Train

The trainer resumes automatically when a checkpoint is present in the output
directory, and stops cleanly once the time budget is reached so the session ends
on its own terms rather than being killed mid-step.

In [ ]:
OUT = Path("/kaggle/working") / f"{LANGUAGE}_nopos"
OUT.mkdir(parents=True, exist_ok=True)

# Seed from a previous session's checkpoint dataset so the run continues instead
# of starting over.
#
# rglob rather than a fixed-depth glob: Kaggle mounts a dataset under its own
# name and may nest the notebook's output folder inside it, so the checkpoint can
# sit at a depth this cannot predict. A fixed pattern that misses simply finds
# nothing - and the failure is silent, because training then starts from step 0
# and looks perfectly healthy for ten hours.
import shutil

resumed_from = None
if not (OUT / "checkpoint_latest.pt").exists():
    for previous in sorted(INPUT.rglob("checkpoint_latest.pt")):
        # Only take a checkpoint belonging to THIS language: attaching both
        # languages' datasets at once would otherwise seed Nepali from Hindi.
        if previous.parent.name != OUT.name:
            continue
        for item in previous.parent.glob("*"):
            shutil.copy2(item, OUT / item.name)
        resumed_from = previous.parent
        break

if resumed_from is not None:
    print(f"seeded from {resumed_from}")
else:
    # State this loudly. A silent fall-through to step 0 wastes a full session,
    # and the log otherwise looks completely normal.
    found = sorted(p.parent.name for p in INPUT.rglob("checkpoint_latest.pt"))
    print("NO CHECKPOINT SEEDED - this run will start from step 0.")
    print(f"   looking for a folder named '{OUT.name}' under /kaggle/input")
    print(f"   checkpoint folders found: {found or 'none'}")
    print("   If you meant to resume, stop now and attach "
          f"lma-bonus-{LANGUAGE}-nopos-checkpoint.")

train_config = TrainConfig(
    batch_size=24,
    gradient_accumulation=4,
    learning_rate=6e-4,
    min_learning_rate=6e-5,
    warmup_steps=500,
    max_steps=12_000,        # keep FIXED across sessions: it sets the decay horizon
    weight_decay=0.1,
    grad_clip=1.0,
    eval_interval=500,
    eval_batches=100,
    checkpoint_interval=1_000,
    log_interval=50,
    seed=20260812,
    amp=True,
)

train_data = TokenBinaryDataset(
    DATA_DIR / f"{LANGUAGE}_train.bin", model_config.block_size, seed=train_config.seed
)
val_data = TokenBinaryDataset(
    DATA_DIR / f"{LANGUAGE}_validation.bin", model_config.block_size, seed=train_config.seed + 1
)

print("train:", f"{train_data.stats()['tokens']:,} tokens")
print("val:  ", f"{val_data.stats()['tokens']:,} tokens")

tokens_per_step = train_config.tokens_per_step * model_config.block_size
print(f"\n{tokens_per_step:,} tokens/step x {train_config.max_steps:,} steps "
      f"= {tokens_per_step * train_config.max_steps / 1e9:.2f}B tokens "
      f"({tokens_per_step * train_config.max_steps / train_data.stats()['tokens']:.2f} epochs)")

trainer = Trainer(model, train_data, val_data, train_config, OUT)
state = trainer.train(max_seconds=TIME_BUDGET_HOURS * 3600)

## 6. Where the run got to

In [ ]:
import math

print(f"step {state.step:,} / {train_config.max_steps:,}")
print(f"tokens seen: {state.tokens_seen / 1e9:.2f}B")
print(f"best val loss: {state.best_val_loss:.4f}  (ppl {math.exp(state.best_val_loss):.2f})")
print(f"wall time: {state.wall_seconds / 3600:.2f} h")

if state.history:
    print("\n   step      val_loss      ppl")
    for row in state.history[-10:]:
        print(f"  {row['step']:>7,}   {row['val_loss']:>8.4f}   {row['val_perplexity']:>8.2f}")

if state.step < train_config.max_steps:
    print("\nNot finished. Save this output as a dataset, attach it to a new")
    print("session alongside lma-phase2-tokenized, and run again to continue.")
else:
    print("\nTraining complete.")

## 7. Loss curve

In [ ]:
import matplotlib.pyplot as plt

if state.history:
    steps = [row["step"] for row in state.history]
    losses = [row["val_loss"] for row in state.history]

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(steps, losses, marker="o", markersize=3)
    axes[0].set_title(f"Validation loss - Model {'H' if LANGUAGE == 'hi' else 'L'}, no positional embeddings")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross-entropy")
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, [math.exp(min(v, 20)) for v in losses], marker="o", markersize=3, color="tab:orange")
    axes[1].set_title("Validation perplexity")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("perplexity")
    axes[1].set_yscale("log")
    axes[1].grid(alpha=0.3)

    figure.tight_layout()
    figure.savefig(OUT / "loss_curve.png", dpi=150)
    plt.show()
else:
    print("no evaluations recorded yet")

## 8. A sample of what it writes

Not a graded metric - just a quick check that the model produces text in the
right script rather than noise.

In [ ]:
prompt = torch.zeros((1, 1), dtype=torch.long, device=trainer.device)
generated = trainer.model.generate(prompt, max_new_tokens=60, temperature=0.8, top_k=50)
print("token ids:", generated[0, :20].tolist(), "...")
print("\n(decode with the Phase 1 tokenizer during evaluation)")

## 9. Save the checkpoint

**Do not skip this.** `/kaggle/working` is wiped when the session ends.

Save the notebook version, then from the **Output** tab create a dataset named
`lma-bonus-<language>-nopos-checkpoint`. Attach it to the next session together with
`lma-phase2-tokenized`, and the run continues from where it stopped.

In [ ]:
for path in sorted(OUT.iterdir()):
    print(f"{path.stat().st_size / 1e6:9.1f} MB  {path.name}")

print(f"\nSave as a Kaggle Dataset named lma-bonus-{LANGUAGE}-nopos-checkpoint")